# Phase 0 + 1 — ยืนยันสเปกโมเดล + วัด Baseline

Notebook นี้ทำ 2 อย่างของ pipeline ([แผนเต็ม](../llm_pruning_project_plan.md)):
- **Phase 0** — โหลด Typhoon 2 3B แล้วยืนยัน `config.json` จริง + นับพารามิเตอร์จริง (ห้ามเชื่อตัวเลขสมมติในแผนจนกว่าจะยืนยัน)
- **Phase 1** — วัด baseline 3 แกนของโมเดล **เดิม** ไว้เป็นจุดอ้างอิง (B1) สำหรับเทียบทุก phase: คุณภาพในโดเมน / นอกโดเมน / ประสิทธิภาพ

## วิธีรันบน Kaggle
1. Settings → Accelerator = **GPU T4 x2** หรือ **P100** (16GB พอสำหรับ 3B ที่ BF16)
2. Settings → Internet = **On** (ต้องโหลดโมเดลจาก Hugging Face)
3. ใส่ HF token ใน **Add-ons → Secrets** ชื่อ `HF_TOKEN` (ถ้าโมเดล gated)
4. รันบนลงล่าง • artifact (ผล baseline) ถูกเซฟไว้ที่ `/kaggle/working/` แล้ว Save Version เพื่อเก็บเป็น output

> ✅ Checklist ที่ปิดได้หลังรัน notebook นี้: Phase 0 ทั้งหมด + Phase 1 (setup, latency, RAM, size, out-of-domain). ส่วน **in-domain** ต้องรอ test set จาก Phase 2

## 0. ติดตั้ง dependencies

Kaggle มี torch/transformers มาให้แล้ว แต่ pin เวอร์ชันให้ reproduce ได้ ปรับตามต้องการ

In [3]:
%pip install -q -U "transformers>=4.44" accelerate bitsandbytes datasets sentencepiece
# lm-eval ติดตั้งแยกเพราะ dependency หนัก; เปิดใช้เมื่อจะรัน harness เต็ม
# %pip install -q lm-eval

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, json, time, gc, platform
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# ===== ตั้งค่าโปรเจกต์ =====
# ยืนยัน model id จริงของ Typhoon 2 3B ก่อน (instruct variant)
MODEL_ID = "scb10x/llama3.2-typhoon2-3b-instruct"
DTYPE = torch.bfloat16
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# HF token จาก Kaggle Secrets (ถ้ามี)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

device: cuda
GPU: Tesla T4
torch: 2.10.0+cu128


## Phase 0 — ยืนยันสเปกโมเดลฐาน

โหลดแค่ `config.json` ก่อน (เบา ไม่ต้องโหลด weight) แล้วเทียบกับตัวเลขที่แผนสมมติไว้ (Llama 3.2 3B)

In [5]:
cfg = AutoConfig.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))

# ค่าที่แผนสมมติไว้ — ใช้ตรวจว่าตรงจริงไหม
PLAN_ASSUMED = {
    "hidden_size": 3072,
    "num_hidden_layers": 28,
    "num_attention_heads": 24,
    "num_key_value_heads": 8,
    "head_dim": 128,
    "intermediate_size": 8192,
    "vocab_size": 128256,
    "tie_word_embeddings": True,
}

print(f"{'key':<24}{'จริง':<14}{'แผนสมมติ':<14}{'ตรง?'}")
print("-" * 60)
for k, assumed in PLAN_ASSUMED.items():
    actual = getattr(cfg, k, "(ไม่มี)")
    ok = "✅" if actual == assumed else "❌ ต่าง!"
    print(f"{k:<24}{str(actual):<14}{str(assumed):<14}{ok}")

print("\n⚠️ ถ้ามี ❌ ให้ไปแก้ตาราง param/RAM ใน llm_pruning_project_plan.md ให้ตรงค่าจริง")

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

key                     จริง          แผนสมมติ      ตรง?
------------------------------------------------------------
hidden_size             3072          3072          ✅
num_hidden_layers       28            28            ✅
num_attention_heads     24            24            ✅
num_key_value_heads     8             8             ✅
head_dim                128           128           ✅
intermediate_size       8192          8192          ✅
vocab_size              128256        128256        ✅
tie_word_embeddings     True          True          ✅

⚠️ ถ้ามี ❌ ให้ไปแก้ตาราง param/RAM ใน llm_pruning_project_plan.md ให้ตรงค่าจริง


In [6]:
# โหลดโมเดลเต็ม (ใช้ต่อใน baseline) — BF16 ~6.4GB
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=device,
    token=os.environ.get("HF_TOKEN"),
)
model.eval()
print("โหลดโมเดลสำเร็จ")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

โหลดโมเดลสำเร็จ


In [7]:
# นับพารามิเตอร์จริง + แยกว่าอยู่ที่ไหน (embedding vs transformer layers)
total = sum(p.numel() for p in model.parameters())

emb = model.get_input_embeddings()
emb_params = emb.weight.numel()
tied = getattr(cfg, "tie_word_embeddings", False)
# ถ้า tie=False ต้องนับ lm_head เพิ่ม
lm_head_params = 0
if not tied and hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
    lm_head_params = model.get_output_embeddings().weight.numel()

embedding_total = emb_params + lm_head_params
transformer_params = total - embedding_total
n_layers = cfg.num_hidden_layers

def b(x):
    return f"{x/1e9:.3f}B ({x/1e6:.1f}M)"

print(f"รวมทั้งหมด          : {b(total)}")
print(f"embedding (tied={tied}): {b(embedding_total)}  = {embedding_total/total*100:.1f}%")
print(f"transformer layers   : {b(transformer_params)}  = {transformer_params/total*100:.1f}%")
print(f"ต่อ 1 layer (เฉลี่ย)  : {b(transformer_params/n_layers)}")
print("\nเทียบแผน: รวม ~3.21B, embedding ~12.3%, transformer ~87.7%")

รวมทั้งหมด          : 3.213B (3212.7M)
embedding (tied=True): 0.394B (394.0M)  = 12.3%
transformer layers   : 2.819B (2818.7M)  = 87.7%
ต่อ 1 layer (เฉลี่ย)  : 0.101B (100.7M)

เทียบแผน: รวม ~3.21B, embedding ~12.3%, transformer ~87.7%


## Phase 1 — Baseline แกนที่ 3: ประสิทธิภาพ (ขนาด / RAM / latency)

In [8]:
# ขนาด weight ในหน่วยความจำตามจำนวน param × bytes/param
bytes_per_param = torch.finfo(DTYPE).bits // 8
weight_gb = total * bytes_per_param / 1e9
print(f"ขนาด weight ({DTYPE}): {weight_gb:.2f} GB  ({bytes_per_param} bytes/param)")

# RAM/VRAM จริงที่ใช้ตอนนี้
if device == "cuda":
    torch.cuda.synchronize()
    print(f"VRAM allocated      : {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"VRAM reserved (peak): {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

ขนาด weight (torch.bfloat16): 6.43 GB  (2 bytes/param)
VRAM allocated      : 6.43 GB
VRAM reserved (peak): 6.43 GB


In [10]:
# Latency / throughput — generate แล้ววัด tokens/วินาที (เฉลี่ยหลายรอบ)
@torch.no_grad()
def benchmark_generation(prompt, max_new_tokens=128, n_runs=3, warmup=1):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    for _ in range(warmup):
        model.generate(**inputs, max_new_tokens=16, do_sample=False)
    times = []
    for _ in range(n_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    gen_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    avg = sum(times) / len(times)
    return {"avg_sec": avg, "gen_tokens": gen_tokens, "tok_per_sec": gen_tokens / avg}

perf = benchmark_generation("อธิบายขั้นตอนการยื่นขอทุนการศึกษาโดยย่อ")
print(f"latency: {perf['avg_sec']:.2f}s / {perf['gen_tokens']} tokens  →  {perf['tok_per_sec']:.1f} tok/s")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


latency: 5.14s / 128 tokens  →  24.9 tok/s


## Phase 1 — Baseline แกนที่ 2: ความสามารถนอกโดเมน (โค้ด/คณิต)

เก็บผลตรงนี้ไว้ **พิสูจน์การถดถอย** ภายหลัง — หลัง specialize ค่าพวกนี้ *ควรตก* (เป็นเป้าหมาย ไม่ใช่บั๊ก) ตอนนี้แค่ probe เชิงคุณภาพ; การวัดเป็นตัวเลขจริงให้ใช้ `lm-eval-harness` (เช่น `gsm8k`, `humaneval`) เมื่อพร้อม

In [14]:
@torch.no_grad()
def chat(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

ood_probes = [
    "Write a Python function that returns the nth Fibonacci number.",
    "ร้านขายส้ม 3 กิโล กิโลละ 25 บาท ทอนจากแบงค์ร้อยเท่าไร?",
]
for p in ood_probes:
    print("Q:", p)
    print("A:", chat(p, max_new_tokens=200)[:500])
    print("-" * 60)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: Write a Python function that returns the nth Fibonacci number.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A: Here is a Python function that returns the nth Fibonacci number:

```python
def fibonacci(n):
    if n <= 0:
        return "Input should be positive integer."
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n):
            a, b = b, a + b
        return b
```

This function uses a loop to calculate the nth Fibonacci number.
------------------------------------------------------------
Q: ร้านขายส้ม 3 กิโล กิโลละ 25 บาท ทอนจากแบงค์ร้อยเท่าไร?
A: แบงค์ร้อยมีค่าเท่ากับ 100 บาท

1. ราคาส้ม = 3 กิโล * 25 บาท/กิโล = 75 บาท
2. ทอนจากแบงค์ร้อย = 100 บาท - 75 บาท = 25 บาท

ดังนั้น ทอนจากแบงค์ร้อยเท่ากับ 25 บาท
------------------------------------------------------------


## Phase 1 — Baseline แกนที่ 1: คุณภาพในโดเมน

⏳ **ต้องรอ test set จาก Phase 2** (`02_dataset.ipynb`) ตอนนี้ทำได้แค่ probe เชิงคุณภาพ เมื่อมี test set แล้วให้ attach เป็น Kaggle Dataset input แล้วรัน cell ด้านล่าง

In [15]:
# probe ชั่วคราว (เปลี่ยนเป็น loop บน test set จริงเมื่อมี Phase 2)
for p in ["หอพักนักศึกษาเปิดให้ลงทะเบียนช่วงไหน?", "What documents are required to apply for a scholarship?"]:
    print("Q:", p)
    print("A:", chat(p, max_new_tokens=200)[:500])
    print("-" * 60)

# --- TEMPLATE สำหรับเมื่อมี test set จริง ---
# from datasets import load_dataset
# test = load_dataset("json", data_files="/kaggle/input/<dataset>/test.jsonl")["train"]
# preds = [chat(ex["question"]) for ex in test]
# คำนวณ metric (เช่น LLM-as-judge / ROUGE / exact-match ตามรูปแบบคำตอบ)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: หอพักนักศึกษาเปิดให้ลงทะเบียนช่วงไหน?


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A: หอพักนักศึกษาเปิดให้ลงทะเบียนในช่วงเปิดเทอมใหม่ โดยทั่วไปจะเริ่มต้นในช่วงเดือนพฤษภาคมหรือมิถุนายนของทุกปี ขึ้นอยู่กับมหาวิทยาลัยและประเภทของหอพักนั้นๆ ควรตรวจสอบกับมหาวิทยาลัยหรือหอพักโดยตรงเพื่อข้อมูลที่ถูกต้องและเป็นปัจจุบัน.
------------------------------------------------------------
Q: What documents are required to apply for a scholarship?
A: To apply for a scholarship, you typically need to provide the following documents:

1. **Application Form**: A completed application form specific to the scholarship you are applying for.

2. **Transcripts**: Official transcripts of your academic records, showing your grades and courses completed.

3. **Resume/CV**: A resume or curriculum vitae that highlights your academic achievements, work experience, and any relevant skills.

4. **Letters of Recommendation**: Typically, two to three letters 
------------------------------------------------------------


## บันทึกผล baseline (B1) เป็น artifact

เซฟเป็น JSON ที่ `/kaggle/working/` แล้ว **Save Version** เพื่อเก็บเป็น output ของ notebook (ใช้เทียบกับ B2/B3 ใน Phase 7)

In [16]:
baseline = {
    "label": "B1_original_typhoon2_3b",
    "model_id": MODEL_ID,
    "dtype": str(DTYPE),
    "config": {k: getattr(cfg, k, None) for k in PLAN_ASSUMED},
    "params": {
        "total": total,
        "embedding": embedding_total,
        "transformer": transformer_params,
        "per_layer_avg": transformer_params // n_layers,
    },
    "efficiency": {
        "weight_gb": round(weight_gb, 3),
        "vram_allocated_gb": round(torch.cuda.memory_allocated()/1e9, 3) if device == "cuda" else None,
        "tok_per_sec": round(perf["tok_per_sec"], 2),
        "gen_latency_sec": round(perf["avg_sec"], 3),
    },
    "in_domain": "TODO: รอ test set Phase 2",
    "out_of_domain": "TODO: รัน lm-eval (gsm8k/humaneval) เพื่อเก็บตัวเลข",
    "env": {"platform": platform.platform(), "torch": torch.__version__},
}

path = os.path.join(OUT_DIR, "baseline_B1.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)
print("เซฟแล้ว:", path)
print(json.dumps(baseline, ensure_ascii=False, indent=2))

เซฟแล้ว: /kaggle/working/baseline_B1.json
{
  "label": "B1_original_typhoon2_3b",
  "model_id": "scb10x/llama3.2-typhoon2-3b-instruct",
  "dtype": "torch.bfloat16",
  "config": {
    "hidden_size": 3072,
    "num_hidden_layers": 28,
    "num_attention_heads": 24,
    "num_key_value_heads": 8,
    "head_dim": 128,
    "intermediate_size": 8192,
    "vocab_size": 128256,
    "tie_word_embeddings": true
  },
  "params": {
    "total": 3212749824,
    "embedding": 394002432,
    "transformer": 2818747392,
    "per_layer_avg": 100669549
  },
  "efficiency": {
    "weight_gb": 6.425,
    "vram_allocated_gb": 6.435,
    "tok_per_sec": 24.91,
    "gen_latency_sec": 5.139
  },
  "in_domain": "TODO: รอ test set Phase 2",
  "out_of_domain": "TODO: รัน lm-eval (gsm8k/humaneval) เพื่อเก็บตัวเลข",
  "env": {
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
    "torch": "2.10.0+cu128"
  }
}


## ✅ สรุปสิ่งที่ปิด checklist ได้ + ขั้นต่อไป

**ปิดได้หลังรัน notebook นี้** (ไปติ๊กใน [`PROGRESS.md`](../PROGRESS.md)):
- Phase 0 — ยืนยัน config + นับ param จริง
- Phase 1 — setup, latency, RAM, ขนาดไฟล์, out-of-domain (probe)

**ยังค้าง:**
- in-domain quality → ทำใน Phase 2 (มี test set ก่อน)
- ตัวเลข out-of-domain เป็นทางการ → รัน `lm-eval-harness`

**ขั้นต่อไป:** `02_dataset.ipynb` — สร้างชุด Q&A กิจการนักศึกษา (เก็บ `test.jsonl` ไว้ย้อนมาวัด in-domain ของ B1 ที่นี่)